# Notebook two. Demonstration

Runs the method end to end, being instance generation, the anticipative MILP benchmark, Method One imitation training, Method Two reinforcement training from the Method One warm start, and evaluation against the two classical baselines.

The default scale completes in about ten minutes on a laptop. The next cell is the only place scale is set, and switching it to `"production"` reproduces the thesis configuration, which takes days. The benchmark needs a Gurobi licence and the preflight cell fails with a clear message if one is not present. Everything is written to `notebooks/scratch/`, and nothing under `cache/`, `checkpoints/`, `results/`, `provenance/` or `figures/` is read or written.

In [ ]:
# "demo" is the reduced scale, about ten minutes. "production" is the thesis configuration and takes days.
SCALE = "demo"

PRESETS = {
    "demo": dict(
        R=4, T=8, H=10, warehouse_size=30.0,
        n_train=40, n_val=8,
        train_seed0=800000, val_seed0=810000,
        mip_time_limit=10, mip_gap=0.02,
        m1_steps=800, m1_batch=16, m1_eval_every=200, m1_ckpt_every=200,
        m2_steps=30, m2_batch=8, m2_rloo_k=4, m2_eval_every=10, m2_ckpt_every=5,
    ),
    "production": dict(
        R=6, T=18, H=20, warehouse_size=50.0,
        n_train=1000, n_val=200,
        train_seed0=10000, val_seed0=11000,
        mip_time_limit=60, mip_gap=0.01,
        m1_steps=10000, m1_batch=32, m1_eval_every=1000, m1_ckpt_every=1000,
        m2_steps=1000, m2_batch=128, m2_rloo_k=20, m2_eval_every=25, m2_ckpt_every=25,
    ),
}
assert SCALE in PRESETS, f"SCALE must be one of {list(PRESETS)}"
P = PRESETS[SCALE]

# The production objective weights. Fixed before the experiments and not recalibrated.
WEIGHTS = {"w_dist": 0.0637, "w_make": 0.2398, "w_bal": 0.6965}

# Seeds sit outside 10000 to 11399 so nothing here touches a reported split.
print(f"SCALE = {SCALE}")
for k, v in P.items():
    print(f"  {k:<16} {v}")
print(f"  weights          {WEIGHTS}")

In [ ]:
# Paths and imports. Everything this notebook writes lives under SCRATCH.
from __future__ import annotations
import json, os, shutil, sys, time
from pathlib import Path
import numpy as np

REPO = Path.cwd()
if not (REPO / "src").is_dir():
    REPO = REPO.parent
assert (REPO / "src").is_dir(), f"cannot locate the repository root from {Path.cwd()}"
sys.path.insert(0, str(REPO / "src"))
sys.path.insert(0, str(REPO / "sweep"))

SCRATCH = REPO / "notebooks" / "scratch" / SCALE
CACHE   = SCRATCH / "cache"
CKPT    = SCRATCH / "checkpoints"
for d in (CACHE / "train", CACHE / "val", CKPT):
    d.mkdir(parents=True, exist_ok=True)

PROTECTED = [REPO / "cache", REPO / "checkpoints", REPO / "results",
             REPO / "provenance", REPO / "figures" / "output"]

def assert_scratch_only(path: Path) -> Path:
    """Refuse any write outside the notebook scratch directory."""
    p = Path(path).resolve()
    if SCRATCH.resolve() not in p.parents and p != SCRATCH.resolve():
        raise RuntimeError(f"REFUSING to write outside the scratch directory: {p}")
    return p

print(f"repository : {REPO}")
print(f"scratch    : {SCRATCH}")
print(f"protected  : {', '.join(str(p.relative_to(REPO)) for p in PROTECTED)}")

In [ ]:
# Gurobi preflight, so a missing licence fails here rather than inside a solve.
def gurobi_preflight() -> str:
    try:
        import gurobipy as gp
    except ImportError as exc:
        raise RuntimeError(
            "Gurobi is not installed in this environment. The anticipative benchmark "
            "is a MILP and needs it. Install with pip install gurobipy, then obtain a "
            "licence. Academic licences are free at https://www.gurobi.com/academia/"
        ) from exc
    try:
        env = gp.Env(empty=True)
        env.setParam("OutputFlag", 0)
        env.start()
        m = gp.Model(env=env)
        x = m.addVar(ub=1.0)
        m.setObjective(x, gp.GRB.MAXIMIZE)
        m.optimize()
        ver = ".".join(str(v) for v in gp.gurobi.version())
        m.dispose(); env.dispose()
    except gp.GurobiError as exc:
        raise RuntimeError(
            "Gurobi is installed but no usable licence was found. The solver reported: "
            f"{exc}. The size-limited pip licence will not solve instances at this "
            "scale. Free academic licences are at https://www.gurobi.com/academia/"
        ) from exc
    return ver

GUROBI_VERSION = gurobi_preflight()
print(f"Gurobi {GUROBI_VERSION}, licence OK")

## 1. Instance generation

In [ ]:
# Generate the instances. No solver yet.
from instances.synthetic_generator import generate_instance, check_seed_non_overlap

INSTANCE_CONFIG = {"R": P["R"], "T": P["T"], "H": P["H"],
                   "warehouse_size": P["warehouse_size"]}

train_seeds = list(range(P["train_seed0"], P["train_seed0"] + P["n_train"]))
val_seeds   = list(range(P["val_seed0"],   P["val_seed0"]   + P["n_val"]))
check_seed_non_overlap(train_seeds, val_seeds)

PRODUCTION_RANGE = set(range(10000, 11400))
overlap = PRODUCTION_RANGE & set(train_seeds + val_seeds)
assert not overlap, f"demo seeds collide with the production splits: {sorted(overlap)[:5]}"

t0 = time.time()
instances = {s: generate_instance(s, INSTANCE_CONFIG) for s in train_seeds + val_seeds}
print(f"generated {len(instances)} instances in {time.time()-t0:.1f}s")

ex = instances[train_seeds[0]]
print(f"\nseed {train_seeds[0]}: R={ex.R}, T={ex.T}, "
      f"arena {ex.config['warehouse_size']}, H={ex.config['H']}, "
      f"Delta={ex.config['Delta']}, v={ex.config['v']}")
print(f"  release epochs : {sorted({int(t['release_epoch']) for t in ex.tasks})}")
print(f"  durations      : min {min(t['duration'] for t in ex.tasks):.2f}, "
      f"max {max(t['duration'] for t in ex.tasks):.2f}")
print("  release epoch is Beta(2,2) mapped onto {0..H-1}; duration is a truncated normal with mean 5.0 and standard deviation 1.5 on [1, 10]")

## 2. The anticipative MILP benchmark

In [ ]:
# Solve the benchmark and extract the per-epoch expert decisions into the scratch cache.
import training.expert_dataset_generator as edg

# Point the generator's cache root at the notebook scratch directory, since it writes to a module-level path.
edg.CACHE_DIR = assert_scratch_only(CACHE)

def build_split(split: str, seeds: list) -> dict:
    stats = {"solved": 0, "optimal": 0, "gaps": [], "seconds": []}
    for i, seed in enumerate(seeds, 1):
        t = time.time()
        rec = edg.process_one_seed(split, seed, config=INSTANCE_CONFIG, weights=WEIGHTS,
                                   mip_time_limit=P["mip_time_limit"], mip_gap=P["mip_gap"])
        dt = time.time() - t
        sol = rec.get("milp_solution", {})
        stats["solved"] += 1
        stats["optimal"] += int(str(sol.get("status", "")).lower() == "optimal")
        if sol.get("mip_gap") is not None:
            stats["gaps"].append(float(sol["mip_gap"]))
        stats["seconds"].append(dt)
        if i % max(1, len(seeds) // 6) == 0 or i == len(seeds):
            print(f"  {split} {i}/{len(seeds)}  last {dt:.1f}s", flush=True)
    return stats

t0 = time.time()
print("solving the benchmark, one MILP per instance")
train_stats = build_split("train", train_seeds)
val_stats   = build_split("val",   val_seeds)
print(f"\ntotal solve wall clock {time.time()-t0:.0f}s")
for name, st in (("train", train_stats), ("val", val_stats)):
    g = np.array(st["gaps"]) if st["gaps"] else np.array([np.nan])
    print(f"  {name}: {st['solved']} solved, {st['optimal']} proven optimal, "
          f"mean residual gap {np.nanmean(g)*100:.1f} %, "
          f"mean {np.mean(st['seconds']):.1f}s per instance")
print("\nEvery record carries the instance, the MILP solution and the per-epoch expert decisions the imitation stage trains on.")

## 3. The Hungarian augmentation

The assignment solved at every decision is over the full robot and task sets, with
availability and pendingness expressed by masking rather than by resizing. The augmenting
blocks give every robot an idle option and every task a wait option, so the assignment is
always feasible and the shape is identical at every decision.

In [ ]:
# Show the augmented matrix and the decode, on one instance, before any training.
import torch
from simulator.dynamic_simulator import DynamicSimulator
from scoring.gnn_scorer import GNNScorer
from evaluation.method_one_evaluator import _decode_action

demo_inst = instances[val_seeds[0]]
sim = DynamicSimulator(demo_inst)
scorer = GNNScorer(num_layers=2)
scorer.eval()

with torch.no_grad():
    state = sim.state
    theta = scorer(state, demo_inst)
R, T = demo_inst.R, demo_inst.T
assert tuple(theta.shape) == (R + T, R + T), "the scorer should return the augmented matrix"

print(f"R = {R} robots, T = {T} tasks, augmented matrix {tuple(theta.shape)}")
print()
print(f"  top left     {R} x {T}   scores, invalid pairs masked at a finite -100")
print(f"  top right    {R} x {R}   idle option, zero on the diagonal, off-diagonal masked")
print(f"  bottom left  {T} x {T}   wait option, zero on the diagonal, off-diagonal masked")
print(f"  bottom right {T} x {R}   zero")
print()
print("A pair is valid when the robot is available and the task is pending. A match in either augmenting block means the robot idles or the task waits, and the action is read from the top left block only.")

with torch.no_grad():
    commits = _decode_action(theta, state, R, T, epsilon=0.0)
print()
print(f"epoch 0: {len(state.available_robots)} robots available, {len(state.pending_tasks)} tasks pending, {len(commits)} commitments {commits}")

## 4. Method One, imitation with the Fenchel-Young loss

Intermediate checkpoints are written every `m1_ckpt_every` steps, so an interrupted run
resumes rather than restarting.

In [ ]:
# Method One training. Writes only into the scratch checkpoint directory.
import training.il_trainer as il

M1_CKPT = assert_scratch_only(CKPT / "method_one.pt")
M1_EVAL_CSV = assert_scratch_only(SCRATCH / "method_one_eval.csv")

m1_cfg = il.TrainingConfig(
    cache_dir=str(CACHE), split="train",
    val_cache_dir=str(CACHE / "val"),
    checkpoint_path=str(M1_CKPT),
    batch_size=P["m1_batch"],
    lr=5e-4, lr_schedule="cosine",
    epsilon_initial=1.0, epsilon_terminal=0.4,
    epsilon_anneal_steps=max(1, P["m1_steps"] // 3),
    num_layers=2,
    checkpoint_every_steps=P["m1_ckpt_every"],
    eval_every_steps=P["m1_eval_every"],
    eval_csv_path=str(M1_EVAL_CSV),
    run_tag=f"demo_{SCALE}",
    log_every_steps=max(1, P["m1_steps"] // 6),
    save_best=True,
    seed=0,
)
t0 = time.time()
m1_history = il.train(m1_cfg, num_steps=P["m1_steps"])
M1_BEST = il.best_checkpoint_path(M1_CKPT)
losses = [h["loss"] for h in m1_history]

print(f"\nMethod One finished in {time.time()-t0:.0f}s over {len(m1_history)} steps, loss {losses[0]:.4f} to {losses[-1]:.4f}")
print(f"  checkpoints: {M1_CKPT.relative_to(REPO)} and {Path(M1_BEST).relative_to(REPO)}")

## 5. Method Two, RLOO from the Method One warm start

The warm start load is the point of this cell. Method Two has no architecture of its own,
it inherits the shape of its warm start, and the thesis result is that the imitation stage
keeps optimisation in a stable regime rather than starting it from a higher level.

In [ ]:
# Method Two through sweep/train_one.py, the documented entry point, which reads sweep/configs.json and resolves config G_v4warmstart.
import subprocess

WARM = Path(M1_BEST) if Path(M1_BEST).exists() else M1_CKPT
assert WARM.exists(), "no Method One checkpoint to warm start from"
before = torch.load(WARM, map_location="cpu", weights_only=False)["model_state_dict"]
print(f"warm starting from {WARM.relative_to(REPO)}, {len(before)} parameter tensors")

M2_CKPT = assert_scratch_only(CKPT / "method_two.pt")
cmd = [
    sys.executable, "-u", str(REPO / "sweep" / "train_one.py"),
    "--config-id", "G_v4warmstart",
    "--cache-dir", str(assert_scratch_only(CACHE / "train")),
    "--val-cache-dir", str(assert_scratch_only(CACHE / "val")),
    "--checkpoint-path", str(M2_CKPT),
    "--warm-start-checkpoint", str(WARM),
    "--max-steps", str(P["m2_steps"]),
    "--batch-size", str(P["m2_batch"]),
    "--rloo-k", str(P["m2_rloo_k"]),
    "--eval-every", str(P["m2_eval_every"]),
    "--checkpoint-every", str(P["m2_ckpt_every"]),
    "--num-workers", "1",
    "--epsilon-end", "0.5",
    "--epsilon-anneal-steps", str(P["m2_steps"]),
]
print("\n" + " ".join(cmd[1:]) + "\n")
env = dict(os.environ, PYTHONPATH=str(REPO / "src"))
t0 = time.time()
proc = subprocess.run(cmd, cwd=str(REPO), env=env, text=True,
                      stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
for line in proc.stdout.splitlines():
    if any(k in line for k in ("[step", "[eval step", "Saved", "Warm-start",
                               "Training method two", "Best gap", "config")):
        print("   ", line)
if proc.returncode != 0:
    print(proc.stdout[-3000:])
    raise RuntimeError(f"sweep/train_one.py exited {proc.returncode}")

after = torch.load(M2_CKPT, map_location="cpu", weights_only=False)["model_state_dict"]
moved = sum(1 for k in before if not torch.equal(before[k], after[k]))
print(f"\nMethod Two finished in {time.time()-t0:.0f}s, {moved} of {len(before)} tensors moved from the imitation weights, checkpoint at {M2_CKPT.relative_to(REPO)}")

## 6. Evaluation against the classical baselines

In [ ]:
# Evaluate the learned policy and both Hungarian baselines on the held-out split.
from baselines.bipartite_policies import run_greedy_policy, run_hungarian_kappa_policy
from evaluation.method_one_evaluator import (
    load_scorer_from_checkpoint, rollout_policy, rollout_failed)

def episode_cost(sim) -> float:
    b = sim.compute_cost(WEIGHTS)
    return float(b.combined if hasattr(b, "combined") else b)

learned = load_scorer_from_checkpoint(M2_CKPT)
learned.eval()

recs = []
for seed in val_seeds:
    inst = instances[seed]
    g = run_greedy_policy(inst)
    k = run_hungarian_kappa_policy(inst, WEIGHTS)
    with torch.no_grad():
        p = rollout_policy(learned, inst, epsilon=0.0)
    cache_file = CACHE / "val" / f"seed{seed}.json"
    milp = json.loads(cache_file.read_text())["milp_solution"]["objective_value"]
    recs.append(dict(seed=seed, greedy=episode_cost(g), kappa=episode_cost(k),
                     policy=episode_cost(p), milp=milp, failed=rollout_failed(p)))

gm = float(np.mean([r["greedy"] for r in recs]))
km = float(np.mean([r["kappa"] for r in recs]))
pm = float(np.mean([r["policy"] for r in recs]))
mm = float(np.mean([r["milp"] for r in recs if r["milp"] is not None]))
served = sum(1 for r in recs if not r["failed"])

print(f"held-out split, {len(recs)} instances, seeds {val_seeds[0]} to {val_seeds[-1]}")
print()
print(f"{'policy':<28} {'mean cost':>10} {'gap closure %':>15}")
for name, m in (("Distance-only Hungarian", gm), ("Kappa-weighted Hungarian", km),
                ("Learned policy, hard decode", pm), ("Anticipative MILP", mm)):
    clo = 100.0 * (gm - m) / (gm - mm) if gm != mm else float("nan")
    print(f"{name:<28} {m:>10.3f} {clo:>15.2f}")
print()
print(f"  learned policy served every task on {served} of {len(recs)} instances")
if SCALE == "demo":
    print("  These are a demonstration that the pipeline runs end to end at reduced scale, not thesis results. Set SCALE to production for those.")